# HW04 深度学习作业
姓名：黄大康
学号：20234080423
日期：2026.06.17

## 2.1 一阶马尔可夫 + 拉普拉斯平滑理论计算
序列：ababc，词汇表 V={a,b,c}, |V|=3
原始转移对：a→b, b→a, a→b, b→c
前驱b统计：count(b→a)=1, count(b→b)=0, count(b→c)=1，总原始转移数=2
拉普拉斯平滑公式：$p(y|x)=\frac{count(x\to y)+1}{\sum_{v\in V}(count(x\to v)+1)}$
分母 = 2 + 3 = 5
1. $p(a|b)=\frac{1+1}{5}=\frac{2}{5}=0.4$
2. $p(c|b)=\frac{1+1}{5}=\frac{2}{5}=0.4$

## 2.2 文本预处理函数 preprocess_text

In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 1.小写，去除标点
    text_low = text.lower()
    clean = [c for c in text_low if c not in string.punctuation]
    text_clean = ''.join(clean)
    # 2.空格分词
    tokens = text_clean.split()
    # 3.按词频降序构建词典，ID从0开始
    word_cnt = Counter(tokens)
    sorted_words = sorted(word_cnt.keys(), key=lambda x: (-word_cnt[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    # 4.滑动窗口生成特征与标签
    feats = []
    labels = []
    max_start = len(tokens) - n
    for i in range(max_start):
        window = tokens[i:i+n]
        next_token = tokens[i + n]
        feats.append(window)
        labels.append(next_token)
    return vocab, (feats, labels)

# 测试
if __name__ == '__main__':
    vocab, (feat_list, lab_list) = preprocess_text("The time machine", n=2)
    print("词汇表:", vocab)
    print("特征序列:", feat_list)
    print("对应标签:", lab_list)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
对应标签: ['machine']


## 3.1 RNN时序反向传播梯度推导
模型：$h_t=W_{hh}h_{t-1}+W_{hx}x_t,\ o_t=W_{oh}h_t,\ L=\frac12\sum_{t=1}^T(o_t-y_t)^2$
1. 单步输出梯度：$\frac{\partial L}{\partial o_t}=o_t-y_t$
2. 隐层瞬时梯度：$\delta_t=\frac{\partial L}{\partial h_t}=W_{oh}^\top(o_t-y_t)+W_{hh}^\top\delta_{t+1},\ \delta_{T+1}=0$
3. $W_{hh}$总梯度：$\frac{\partial L}{\partial W_{hh}}=\sum_{t=2}^T \delta_t h_{t-1}^\top$

### 梯度消失/爆炸条件
梯度递推包含矩阵连乘 $\prod W_{hh}^\top$：
- $W_{hh}$谱半径 < 1：梯度指数衰减 → 梯度消失
- $W_{hh}$谱半径 > 1：梯度指数放大 → 梯度爆炸

## 3.2 RNN单步前向+反向传播实现

In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    pre_act = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(pre_act)
    return h_t, pre_act

def rnn_backward(dh_next, h_t, pre_act, x_t, h_prev, W_hx, W_hh):
    dpre = dh_next * (1 - np.square(h_t))
    db_h = np.sum(dpre, axis=0)
    dW_hx = x_t.T @ dpre
    dW_hh = h_prev.T @ dpre
    dx_t = dpre @ W_hx.T
    dh_prev = dpre @ W_hh.T
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
if __name__ == '__main__':
    B, D_in, H_hid = 2, 3, 4
    x = np.random.randn(B, D_in)
    h_prev = np.zeros((B, H_hid))
    Whx = np.random.randn(D_in, H_hid)
    Whh = np.random.randn(H_hid, H_hid)
    bh = np.zeros(H_hid)
    ht, pre = rnn_forward(x, h_prev, Whx, Whh, bh)
    dh_up = np.random.randn(B, H_hid)
    dx, dhp, dWhx, dWhh, dbh = rnn_backward(dh_up, ht, pre, x, h_prev, Whx, Whh)
    print("h_t shape:", ht.shape)
    print("dW_hh shape:", dWhh.shape)

h_t shape: (2, 4)
dW_hh shape: (4, 4)


## 4.1 深度双向RNN总参数量
参数说明：L层数，每层隐藏单元H，输入维度D，输出维度O
单层单向RNN参数：$DH + H^2 + H$
单层双向RNN参数：$2(DH + H^2 + H)$
L层双向循环总参数：$2L(DH+H^2+H)$
最后输出层（拼接2H维输入）：$2HO + O$
全部参数表达式：
$$P=2L(DH+H^2+H)+2HO+O$$

## 4.2 双向RNN编码器

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, bidirectional=True, batch_first=False)

    def forward(self, X):
        seq_out, hn = self.rnn(X)
        h_fwd = hn[0]
        h_bwd = hn[1]
        final_seq_repr = torch.cat([h_fwd, h_bwd], dim=-1)
        return seq_out, final_seq_repr

# 测试
if __name__ == '__main__':
    slen, batch, indim = 10, 4, 5
    hid = 6
    model = BiRNNEncoder(indim, hid)
    x_in = torch.randn(slen, batch, indim)
    seq_output, final_vec = model(x_in)
    print("时序拼接隐层shape:", seq_output.shape)
    print("整条序列表征shape:", final_vec.shape)

时序拼接隐层shape: torch.Size([10, 4, 12])
整条序列表征shape: torch.Size([4, 12])


## 5.1 Skip-gram负采样损失推导
中心词向量$v_c$，真实上下文$u_o$，K个负样本$u_{n_k}$，$\sigma(z)=\frac{1}{1+e^{-z}}$
最大化对数似然：$\log\sigma(v_c^\top u_o)+\sum_{k=1}^K\log\sigma(-v_c^\top u_{n_k})$
最小化损失（负对数似然）：
$$\mathcal{J}=-\left[\log\sigma(v_c^\top u_o)+\sum_{k=1}^K\log\sigma(-v_c^\top u_{n_k})\right]$$
### 负样本采样规则
使用词频的3/4次幂构建噪声分布，高频词汇采样概率更高，过滤极低频停用词。

## 5.2 CBOW完整Softmax损失实现

In [4]:
import numpy as np

def cbow_forward_loss(context_batch, target_batch, W, W_out):
    total_loss = 0.0
    batch_size = len(context_batch)
    for ctx_ids, tgt in zip(context_batch, target_batch):
        ctx_embeds = W[ctx_ids]
        h = np.mean(ctx_embeds, axis=0)
        logits = h @ W_out
        max_log = np.max(logits)
        exp_log = np.exp(logits - max_log)
        prob = exp_log / np.sum(exp_log)
        loss_single = -np.log(prob[tgt] + 1e-8)
        total_loss += loss_single
    return total_loss / batch_size

# 测试
if __name__ == '__main__':
    V, d, ctx_num = 10, 4, 3
    W_emb = np.random.randn(V, d)
    W_out = np.random.randn(d, V)
    batch_ctx = [[0,1,2], [2,3,4]]
    batch_tgt = [1, 3]
    loss_val = cbow_forward_loss(batch_ctx, batch_tgt, W_emb, W_out)
    print("CBOW批量平均损失:", loss_val)

CBOW批量平均损失: 3.520494554994149


## 6.1 缩放点积注意力计算步骤
$Q\in\mathbb{R}^{2\times4}, K\in\mathbb{R}^{3\times4}, V\in\mathbb{R}^{3\times5}, d_k=4,\sqrt{d_k}=2$
1. 原始得分矩阵：$S=\frac{QK^\top}{\sqrt{d_k}} \in \mathbb{R}^{2\times3}$
2. 逐行Softmax归一化得到注意力权重矩阵 $A\in\mathbb{R}^{2\times3}$
3. 加权求和输出：$Attn=A \cdot V \in \mathbb{R}^{2\times5}$
输出维度：2行5列

## 6.2 多头注意力 num_heads=2, d_model=4

In [5]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.dk = d_model // num_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def scaled_dot_product_attn(self, q, k, v):
        score = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        attn_weight = torch.softmax(score, dim=-1)
        return attn_weight @ v

    def forward(self, X):
        seq_len, batch, _ = X.shape
        Q = self.wq(X)
        K = self.wk(X)
        V = self.wv(X)
        # 分头 (seq_len, batch, head, dk) -> (seq_len, head, batch, dk)
        Q = Q.view(seq_len, batch, self.num_heads, self.dk).transpose(1, 2)
        K = K.view(seq_len, batch, self.num_heads, self.dk).transpose(1, 2)
        V = V.view(seq_len, batch, self.num_heads, self.dk).transpose(1, 2)
        attn_out = self.scaled_dot_product_attn(Q, K, V)
        # 拼接多头
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(seq_len, batch, self.d_model)
        output = self.out_proj(attn_out)
        return output

# 测试
if __name__ == '__main__':
    mha_layer = MultiHeadAttention(d_model=4, num_heads=2)
    seq, bsz = 6, 3
    x_input = torch.randn(seq, bsz, 4)
    out = mha_layer(x_input)
    print("多头注意力输出shape:", out.shape)

多头注意力输出shape: torch.Size([6, 3, 4])
